In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential, Model
from tensorflow.keras.layers import Dense, Input, Embedding, LayerNormalization, Dropout
import numpy as np

In [ ]:
# Load and preprocess data
with open('training_data.txt', 'r', encoding='utf-8') as f:
    data = f.read().replace('\n', ' ')

In [ ]:
print("Data length:", len(data))
characters = list(set(list(data)))
print("Number of unique characters:", len(characters))

In [ ]:
# Create encoding dictionaries
character_to_integer_encoding = {}
integer_to_character_encoding = {}
for i in range(len(characters)):
    character_to_integer_encoding[characters[i]] = i + 1
    integer_to_character_encoding[i + 1] = characters[i]

def encode(string):
    return [character_to_integer_encoding[char] for char in string]

def decode(lst):
    return ''.join([integer_to_character_encoding[i] for i in lst])

In [ ]:
# Prepare data: 90% training, 10% testing
input_data = encode(data)
train_data = input_data[:int(0.9 * len(input_data))]
test_data = input_data[int(0.9 * len(input_data)):]

In [ ]:
# Hyperparameters
batch_size = 32
block_size = 128  # Sequence length for each training sample
num_heads = 8
num_transformer_blocks = 4
input_vocab_size = len(characters) + 1  # +1 for padding/indexing starting at 1
feed_forward_dim = 256

In [ ]:
# Custom MultiHeadAttention (alternative implementation, not working well)
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, num_heads, model_dimension):
        super().__init__()
        self.num_heads = num_heads
        self.model_dimension = model_dimension
        assert model_dimension % num_heads == 0

        self.depth = model_dimension // num_heads
        self.query_space_projector = Dense(model_dimension)
        self.key_space_projector = Dense(model_dimension)
        self.value_space_projector = Dense(model_dimension)
        self.dense = Dense(model_dimension)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def causal_attention_mask(self, batch_size, n_dest, n_src):
        i = tf.range(n_dest)[:, None]
        j = tf.range(n_src)
        m = i >= j
        mask = tf.cast(m, tf.bool)
        mask = tf.reshape(mask, [1, n_dest, n_src])
        mask = tf.tile(mask, [batch_size, 1, 1])
        mask = mask[:, tf.newaxis, :, :]
        mask = tf.tile(mask, [1, self.num_heads, 1, 1])
        return mask

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        q = self.query_space_projector(inputs)
        k = self.key_space_projector(inputs)
        v = self.value_space_projector(inputs)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        mask = self.causal_attention_mask(batch_size, tf.shape(inputs)[1], tf.shape(inputs)[1])
        mask = tf.cast(mask, tf.float32)
        attention, attention_weights = self.scaled_dot_product_attention(q, k, v, mask)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        attention = tf.reshape(attention, (batch_size, -1, self.model_dimension))
        output = self.dense(attention)
        return output

    def scaled_dot_product_attention(self, q, k, v, mask):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        scaled_attention_logits += (mask * -1e9)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        return output, attention_weights

In [ ]:
# Standalone causal attention mask function
def causal_attention_mask(batch_size, n_dest, n_src):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, tf.bool)
    mask = tf.reshape(mask, [1, n_dest, n_src])  # Completed: reshaped mask to [1, n_dest, n_src]
    return tf.tile(mask, [batch_size, 1, 1])

In [ ]:
# Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        # Multi-head attention layer (using built-in for performance)
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
        # Feed-forward network
        self.ffn = Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim)
        ])
        self.normalization_layer_1 = LayerNormalization(epsilon=1e-6)
        self.normalization_layer_2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False):
        # Self-attention with a causal mask to preserve autoregressive property
        attn_output = self.att(inputs, inputs, use_causal_mask=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.normalization_layer_1(inputs + attn_output)
        # Feed-forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.normalization_layer_2(out1 + ffn_output)

In [ ]:
# Token and Position Embedding
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_embedding = Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_embedding(positions)
        x = self.token_embedding(x)
        return x + positions

In [ ]:
# Transformer Model (Subclass API)
class Transformer(Model):
    def __init__(self, maxlen, vocab_size, embed_dim, num_heads, feed_forward_dim, num_transformer_blocks):
        super().__init__()
        self.inputs = Input(shape=(maxlen,), dtype=tf.int32)
        self.embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
        self.embedding_dim = embed_dim
        self.num_transformer_blocks = num_transformer_blocks
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, feed_forward_dim)
            for _ in range(num_transformer_blocks)
        ]
        self.dense = Dense(vocab_size)

    def call(self, inputs, training=False):
        x = self.embedding_layer(inputs)
        for i in range(self.num_transformer_blocks):
            x = self.transformer_blocks[i](x, training=training)
        output = self.dense(x)1
        return output

In [ ]:
'''
Functional API-based Transformer Model
This version uses the functional API which leverages TensorFlow's static graph optimizations.
'''
def get_transformer_model(maxlen, vocab_size, embed_dim, num_heads, feed_forward_dim, num_transformer_blocks=1):
    inputs = Input(shape=(maxlen,), dtype=tf.int32)
    embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
    x = embedding_layer(inputs)
    for _ in range(num_transformer_blocks):
        transformer_block = TransformerBlock(embed_dim, num_heads, feed_forward_dim)
        x = transformer_block(x)
    outputs = Dense(vocab_size)(x)
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
# Instantiate model using the Functional API version
model = get_transformer_model(block_size, input_vocab_size, feed_forward_dim, num_heads, feed_forward_dim, num_transformer_blocks)

In [ ]:
# Compile the model
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])

In [ ]:
# Prepare training dataset: create input-target pairs
inputs_list = [train_data[i:i + block_size] for i in range(0, len(train_data) - block_size - 1)]
targets_list = [train_data[i + 1:i + block_size + 1] for i in range(0, len(train_data) - block_size - 1)]

dataset = tf.data.Dataset.from_tensor_slices((inputs_list, targets_list))
dataset = dataset.shuffle(10000)
dataset = dataset.batch(batch_size, drop_remainder=True)

In [ ]:
model.summary()

In [ ]:
model.fit(dataset, epochs=10)

In [ ]:
def generate_text(model, start_index, num_generate=1):
    """
    Generates text by predicting one character at a time.
    """
    input_sequence = train_data[start_index:start_index + block_size]
    generated_text = decode(input_sequence)
    exact_sequence = decode(input_sequence)
    for i in range(num_generate):
        input_eval = tf.convert_to_tensor([input_sequence], dtype=tf.int32)
        predictions = model.predict(input_eval)
        probabilities = tf.nn.softmax(predictions[0, -1]).numpy()
        next_token = np.random.choice(len(probabilities), p=probabilities)
        input_sequence += [next_token]
        input_sequence = input_sequence[1:]
        exact_sequence += decode([np.argmax(probabilities)])
        generated_text += decode([next_token])
    return generated_text, exact_sequence

In [ ]:
# Generate sample text
generated_text, exact_sequence = generate_text(model, start_index=0, num_generate=1000)
print("Generated Text:\n", generated_text)